In [1]:
# simulated_customer_data.py
import numpy as np
import pandas as pd

def generate_simulated_customers(n_customers=2000, max_steps=12, seed=42):
    rng = np.random.RandomState(seed)
    rows = []
    for cid in range(n_customers):
        # customer base characteristics (static)
        age = rng.randint(18, 75)
        region = rng.choice(['north','south','east','west'])
        base_value = rng.exponential(scale=100.0)  # latent CLV proxy

        # create a sequence of events (monthly steps)
        recency = rng.randint(0, 365)  # days since last purchase
        freq = rng.randint(0, 10)
        monetary = rng.exponential(50.0)
        for t in range(rng.randint(4, max_steps)):  # variable-length trajectories
            # dynamic features
            recency = max(0, recency + rng.randint(20, 40))  # time moves forward
            freq += rng.binomial(1, 0.05 + 0.01*freq)  # small prob to buy
            monetary = max(1.0, monetary + rng.normal(0, 10))
            # simple state vector
            state = {
                'customer_id': cid,
                'step': t,
                'age': age,
                'region': region,
                'recency_days': recency,
                'frequency': freq,
                'avg_monetary': monetary
            }
            # historical policy: simple rule-based policy that mails if recency large and freq low
            action = 1 if (recency > 120 and freq < 3) else 0  # 1 = send mail, 0 = no mail
            # reward: if mailed, chance to donate proportional to base_value and inverse recency
            p_buy = 0.02 * (base_value / (1+recency/30.0)) + 0.01*(action)
            bought = rng.rand() < p_buy
            donation = float(rng.exponential(scale=base_value/10.0)) if bought else 0.0
            # mailing cost
            cost = 0.5 if action==1 else 0.0
            profit = donation - cost
            row = {**state, 'action': action, 'donation': donation, 'profit': profit}
            rows.append(row)
    df = pd.DataFrame(rows)
    # sort for convenience (customer, step)
    df = df.sort_values(['customer_id','step']).reset_index(drop=True)
    return df

# Example usage:
if __name__ == "__main__":
    df = generate_simulated_customers(500, max_steps=10, seed=123)
    print(df.head())
    df.to_csv("simulated_customers_rl.csv", index=False)


   customer_id  step  age region  recency_days  frequency  avg_monetary  \
0            0     0   63   east           119          6     39.229991   
1            0     1   63   east           154          6     34.940864   
2            0     2   63   east           178          6     27.246530   
3            0     3   63   east           205          6     33.013990   
4            0     4   63   east           241          6     25.237441   

   action  donation  profit  
0       0       0.0     0.0  
1       0       0.0     0.0  
2       0       0.0     0.0  
3       0       0.0     0.0  
4       0       0.0     0.0  


In [2]:
# simple_customer_env.py
import gym
from gym import spaces
import numpy as np
import pandas as pd

class SimpleCustomerEnv(gym.Env):
    """
    A minimal Gym environment for customer mailing decisions.
    - state: vector [recency_days, frequency, avg_monetary, age_code]
    - actions: 0=no mail, 1=send mail
    - reward: profit (donation - cost)
    Episodes: one customer trajectory (multiple steps)
    """
    metadata = {'render.modes': ['human']}

    def __init__(self, customer_df):
        super().__init__()
        # observation: numeric vector (recency, freq, avg_monetary, age_norm)
        self.observation_space = spaces.Box(low=0.0, high=1e6, shape=(4,), dtype=np.float32)
        # action: 0 or 1
        self.action_space = spaces.Discrete(2)
        self.df = customer_df
        # create list of unique customers
        self.customers = customer_df['customer_id'].unique().tolist()
        self.cur_idx = 0
        self.cur_customer = None
        self.step_idx = 0

    def reset(self):
        # pick next customer (cyclic)
        cid = self.customers[self.cur_idx % len(self.customers)]
        self.cur_customer = self.df[self.df['customer_id'] == cid].reset_index(drop=True)
        self.step_idx = 0
        self.cur_idx += 1
        obs = self._get_obs()
        return obs

    def _get_obs(self):
        row = self.cur_customer.loc[self.step_idx]
        age_code = (row['age'] - 18) / 60.0  # normalize age roughly
        return np.array([row['recency_days'], row['frequency'], row['avg_monetary'], age_code], dtype=np.float32)

    def step(self, action):
        row = self.cur_customer.loc[self.step_idx]
        # simple dynamics: if action==1, increase chance to donate
        base_prob = 0.01 * (1.0 / (1 + row['recency_days']/30.0)) + 0.001*row['frequency']
        if action == 1:
            base_prob += 0.02
        bought = np.random.rand() < base_prob
        donation = np.random.exponential(scale=max(1.0, row['avg_monetary']/10.0)) if bought else 0.0
        cost = 0.5 if action==1 else 0.0
        reward = donation - cost
        done = (self.step_idx >= len(self.cur_customer) - 1)
        # advance internal step if not done
        if not done:
            self.step_idx += 1
            obs = self._get_obs()
        else:
            obs = np.zeros(self.observation_space.shape, dtype=np.float32)
        info = {'donation': donation}
        return obs, float(reward), bool(done), info

    def render(self, mode='human'):
        pass

# Example usage:
if __name__ == "__main__":
    from simulated_customer_data import generate_simulated_customers
    df = generate_simulated_customers(200, max_steps=8)
    env = SimpleCustomerEnv(df)
    obs = env.reset()
    total = 0.0
    done = False
    while not done:
        action = env.action_space.sample()  # random policy
        obs, r, done, info = env.step(action)
        total += r
    print("Episode reward:", total)


ModuleNotFoundError: No module named 'gym'

In [ ]:
pip install gym
